# RAG and tool calling

Run one cell at a time and narrate each step. This notebook tells one story:
we start with plain documents, make them searchable, let an LLM answer from
what we found, and finally let the model decide *when* to search.

**The mental model to keep repeating:**

1. RAG is not magic memory. It is **search + a prompt**.
2. **Retrieval** picks the relevant context.
3. **Generation** answers *from that context*, not from memory.
4. **Tool calling** flips control: the model decides when to retrieve or run code.

**Roadmap**

- **Part 0 — Setup:** install deps, connect to LM Studio.
- **Part 1 — Build a searchable knowledge base:** documents → vectors → index → search.
- **Part 2 — Add the LLM (this is RAG):** answer grounded in retrieved context.
- **Part 3 — Tool calling:** the model calls functions, including RAG-as-a-tool.
- **Appendix — Embedding model size:** how the retrieval model changes the results.


## Part 0 — Setup

Install the few libraries the notebook needs. Safe to re-run — it is quiet and
quick if they are already present.


In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "chromadb>=1.5.7",
    "openai>=2.32.0",
    "python-dotenv>=1.2.2",
])


### Connect to LM Studio

Before running the model-backed cells, start LM Studio's local server and load:

- Chat model: `qwen/qwen3.5-9b`
- Embedding model: `text-embedding-qwen3-embedding-4b`

The notebook lives at the demo root and reads documents from `documents/`. It
loads `.env` from the demo root if present, then from `rag_demo/.env`.


In [ ]:
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

import chromadb
from chromadb.api.types import Documents, EmbeddingFunction, Embeddings
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "documents").exists() and (PROJECT_DIR.parent / "documents").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DOCS_DIR = PROJECT_DIR / "documents"
load_dotenv(PROJECT_DIR / ".env")
load_dotenv(PROJECT_DIR / "rag_demo" / ".env")


@dataclass(frozen=True)
class Config:
    openai_base_url: str = os.getenv("OPENAI_BASE_URL", "http://127.0.0.1:1234/v1")
    openai_api_key: str = os.getenv("OPENAI_API_KEY", "lm-studio")
    chat_model: str = os.getenv("CHAT_MODEL", "qwen/qwen3.5-9b")
    embedding_model: str = os.getenv("EMBEDDING_MODEL", "text-embedding-qwen3-embedding-4b")
    top_k: int = int(os.getenv("TOP_K", "3"))


cfg = Config()
client = OpenAI(base_url=cfg.openai_base_url, api_key=cfg.openai_api_key)

print(f"Project directory : {PROJECT_DIR}")
print(f"Documents         : {DOCS_DIR}")
print(f"LM Studio URL     : {cfg.openai_base_url}")
print(f"Chat model        : {cfg.chat_model}")
print(f"Embedding model   : {cfg.embedding_model}")
print(f"Top-K             : {cfg.top_k}")

View available models

In [ ]:
models = client.models.list()
print("Models visible to the OpenAI-compatible endpoint:")
for model in models.data:
    print("-", model.id)

## Part 1 — Build a searchable knowledge base

Goal: turn a folder of markdown files into something we can search *by meaning*.
No LLM yet — just documents, vectors, and a similarity search.


### Step 1 — Load the documents

One markdown file = one chunk, so the retrieval step is easy to see on stage.
These are plain, inspectable files — nothing is hidden.


In [ ]:
def load_markdown_files(directory: Path) -> list[dict[str, str]]:
    docs = []
    for path in sorted(directory.glob("*.md")):
        docs.append({"id": path.stem, "source": path.name, "text": path.read_text(encoding="utf-8")})
    if not docs:
        raise FileNotFoundError(f"No markdown files found in {directory}")
    return docs


docs = load_markdown_files(DOCS_DIR)
print(f"Loaded {len(docs)} documents")
for doc in docs:
    first_line = doc["text"].splitlines()[0]
    print(f"- {doc['source']}: {first_line}")

Peek at one document — this is just text, the raw material for everything that
follows.


In [ ]:
# Pick one document and show that the source text is plain, inspectable data.
display(Markdown(docs[0]["text"]))

### Step 2 — Turn text into vectors

An **embedding** is a list of numbers that captures meaning. Similar meanings
land near each other in vector space. The one rule that makes RAG work: **use
the same embedding model for documents and questions.**


In [ ]:
sample_embedding = client.embeddings.create(
    model=cfg.embedding_model,
    input="What is retrieval augmented generation?",
).data[0].embedding

print(f"Embedding dimensions: {len(sample_embedding)}")
print("First 8 values:", [round(x, 4) for x in sample_embedding[:8]])

### Step 3 — Index the documents

Embed every document and store the vectors in an in-memory Chroma collection.
That collection is our searchable index.


In [ ]:
class OpenAIEmbeddingFunction(EmbeddingFunction[Documents]):
    def __init__(self, openai_client: OpenAI, model: str) -> None:
        self._client = openai_client
        self._model = model

    def __call__(self, inputs: Documents) -> Embeddings:
        response = self._client.embeddings.create(model=self._model, input=list(inputs))
        return [item.embedding for item in response.data]


embedder = OpenAIEmbeddingFunction(client, cfg.embedding_model)
chroma = chromadb.EphemeralClient()
collection = chroma.get_or_create_collection(
    name="devtalks_notebook_rag_demo",
    embedding_function=embedder,
)

collection.upsert(
    ids=[doc["id"] for doc in docs],
    documents=[doc["text"] for doc in docs],
    metadatas=[{"source": doc["source"]} for doc in docs],
)

print(f"Indexed {collection.count()} documents in an in-memory Chroma collection")

### Step 4 — Retrieve (search only, no LLM)

Nothing has been generated yet. We embed the question, find the nearest
documents, and look at what comes back. This is the "R" in RAG, on its own.


In [ ]:
def retrieve(question: str, k: int = cfg.top_k) -> list[dict[str, Any]]:
    result = collection.query(query_texts=[question], n_results=k)
    hits = []
    for text, metadata, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append({
            "text": text,
            "source": metadata["source"],
            "distance": distance,
        })
    return hits


question = "What embedding model does this demo use and why?"
hits = retrieve(question)

print("Question:", question)
print("\nNearest documents:")
for index, hit in enumerate(hits, start=1):
    preview = hit["text"].strip().replace("\n", " ")[:120]
    print(f"{index}. {hit['source']} distance={hit['distance']:.4f} - {preview}...")

In [ ]:
not_in_kb_q = "Who is Rusu Dinu?"
not_in_kb_hits = retrieve(not_in_kb_q)
print("\nNearest documents:")
for index, not_in_kb_hits in enumerate(not_in_kb_hits, start=1):
    preview = not_in_kb_hits["text"].strip().replace("\n", " ")[:120]
    print(f"{index}. {not_in_kb_hits['source']} distance={not_in_kb_hits['distance']:.4f} - {preview}...")

**Talking point:** the model has not seen anything yet. Retrieval quality is
decided *here* — if the wrong documents come back, no amount of clever prompting
will fix the final answer.


## Part 2 — Add the LLM

Now we hand the retrieved passages to the chat model and ask it to answer
**only** from that context. Retrieval + a grounded prompt = RAG.


### Step 5 — Generate an answer grounded in retrieved context

The system prompt enforces two habits: answer only from the provided context,
and cite the source filenames. Watch the citations line up with what Step 4
retrieved.


In [ ]:
def build_grounded_messages(question: str, hits: list[dict[str, Any]]) -> list[dict[str, str]]:
    context = "\n\n".join(f"[source: {hit['source']}]\n{hit['text']}" for hit in hits)
    return [
        {
            "role": "system",
            "content": (
                "You are a precise assistant. Answer using ONLY the provided context. "
                "If the answer is not in the context, say you do not know. "
                "Cite sources by filename."
            ),
        },
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]


def answer_with_rag(question: str, k: int = cfg.top_k) -> str:
    hits = retrieve(question, k=k)
    response = client.chat.completions.create(
        model=cfg.chat_model,
        messages=build_grounded_messages(question, hits),
        temperature=0.2,
    )
    return response.choices[0].message.content or ""


answer = answer_with_rag(question)
display(Markdown(answer))

### Step 6 — The whole RAG pipeline as one function

`answer_with_rag` is the entire non-agentic path: retrieve first, then ask the
model. The **application** decides to retrieve every time. The last question has
no answer in the knowledge base — watch the model say "I don't know" instead of
inventing one.


In [ ]:
for q in [
    "What is DevTalks and what is its official site?",
    "What are the three stages of RAG?",
    "What city hosted the first moon colony in this knowledge base?",
]:
    print("=" * 88)
    print("Question:", q)
    print(answer_with_rag(q))

## Part 3 — Tool calling: let the model decide

So far the application always retrieves first. Tool calling flips that: we hand
the model a set of tools and let it choose when to call them. The loop is always
the same — the model asks, the app runs the tool, the result goes back, the
model continues.


### Step 7 — Tool calling with plain Python functions

Start with two trivial tools, `add` and `current_time`, so the mechanism is
obvious before we point it at retrieval. Each tool is a Python function plus a
JSON schema the model can read.


In [ ]:
def add(a: int, b: int) -> int:
    return a + b


def current_time(timezone: str = "UTC") -> str:
    try:
        tz = ZoneInfo(timezone)
    except ZoneInfoNotFoundError:
        return f"Unknown timezone: {timezone!r}"
    return datetime.now(tz).isoformat(timespec="seconds")


basic_tool_functions = {
    "add": add,
    "current_time": current_time,
}

basic_tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "add",
            "description": "Return the sum of two integers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "a": {"type": "integer"},
                    "b": {"type": "integer"},
                },
                "required": ["a", "b"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "current_time",
            "description": "Return the current wall-clock time in an IANA timezone.",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {
                        "type": "string",
                        "description": "Example: UTC, Europe/Bucharest, America/New_York",
                    }
                },
                "required": ["timezone"],
            },
        },
    },
]

print(json.dumps(basic_tool_schemas, indent=2))

The tool-calling loop: send the question with the tool schemas, run whatever
tool the model asks for, feed the result back, and repeat until the model gives
a final answer.


In [ ]:
def run_tool_calling_agent(
    question: str,
    tool_schemas: list[dict[str, Any]],
    tool_functions: dict[str, Any],
    system_prompt: str,
    max_steps: int = 5,
) -> str:
    messages: list[dict[str, Any]] = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    for step in range(1, max_steps + 1):
        response = client.chat.completions.create(
            model=cfg.chat_model,
            messages=messages,
            tools=tool_schemas,
            tool_choice="auto",
            temperature=0.2,
        )
        message = response.choices[0].message
        tool_calls = message.tool_calls or []

        if not tool_calls:
            return message.content or ""

        print(f"Step {step}: model requested {len(tool_calls)} tool call(s)")
        messages.append({
            "role": "assistant",
            "content": message.content or "",
            "tool_calls": [
                {
                    "id": call.id,
                    "type": "function",
                    "function": {
                        "name": call.function.name,
                        "arguments": call.function.arguments,
                    },
                }
                for call in tool_calls
            ],
        })

        for call in tool_calls:
            name = call.function.name
            try:
                arguments = json.loads(call.function.arguments or "{}")
            except json.JSONDecodeError:
                arguments = {}
            print(f"  calling {name}({arguments})")
            if name not in tool_functions:
                result = f"Unknown tool: {name}"
            else:
                result = tool_functions[name](**arguments)
            print(f"  result: {result}")
            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": str(result),
            })

    return "Stopped before the model produced a final answer."


Ask one question that needs **both** tools and watch the model sequence the
calls:


In [ ]:
tool_answer = run_tool_calling_agent(
    "What is 17 plus 25, and what time is it now in Europe/Bucharest?",
    basic_tool_schemas,
    basic_tool_functions,
    system_prompt=(
        "You are a concise assistant. Use tools when they are useful, "
        "then give a short natural-language answer."
    ),
)

display(Markdown(tool_answer))

### Step 8 — RAG as a tool

Now the retrieval function becomes a tool named `search_knowledge`. Same loop as
Step 7 — only the tool changed. This is the bridge to the MCP demo: MCP is just
a standard way to expose tools like this to other processes, but the
tool-calling idea is identical.


In [ ]:
def search_knowledge(query: str, k: int = cfg.top_k) -> str:
    hits = retrieve(query, k=k)
    blocks = []
    for hit in hits:
        blocks.append(f"[source: {hit['source']}] (distance={hit['distance']:.4f})\n{hit['text'].strip()}")
    return "\n\n---\n\n".join(blocks)


rag_tool_functions = {"search_knowledge": search_knowledge}
rag_tool_schemas = [
    {
        "type": "function",
        "function": {
            "name": "search_knowledge",
            "description": "Search the local DevTalks demo knowledge base and return relevant passages with source filenames.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "Natural-language search query."},
                    "k": {"type": "integer", "description": "Number of passages to return.", "default": cfg.top_k},
                },
                "required": ["query"],
            },
        },
    }
]

print(search_knowledge("What are the three stages of RAG?", k=2)[:1000])

The model now decides for itself to search the knowledge base, then answers with
citations:


In [ ]:
rag_tool_answer = run_tool_calling_agent(
    "What is DevTalks, and what are the three stages of RAG? Cite sources.",
    rag_tool_schemas,
    rag_tool_functions,
    system_prompt=(
        "You are a grounded assistant. When the answer could be in the local knowledge base, "
        "call search_knowledge first. Answer using only retrieved passages and cite filenames. "
        "If the retrieved passages do not contain the answer, say you do not know."
    ),
)

display(Markdown(rag_tool_answer))

### Where this goes next

- Part 1 showed retrieval is just **search**.
- Part 2 showed RAG is **search + a grounded prompt**.
- Part 3 showed tool calling lets the **model** decide when to search.

The MCP demos take this last step further: they move `search_knowledge` out of
the notebook and behind a standard protocol, so any MCP-aware client (like
LM Studio) can call the exact same tool.


## Appendix — Does embedding model size matter?

Optional deep-dive.

Embedding model size affects retrieval quality, latency, and memory. A larger
model often separates subtle meanings better, but it costs more to run. This
indexes the same documents twice — once with the 4B model and once with
`qwen3-0.6b-text-embedding` — and compares the retrieval. The chat model never
changes; only the retrieval model does.


In [ ]:
import time

EMBEDDING_MODELS_TO_COMPARE = [
    cfg.embedding_model,
    os.getenv("SMALL_EMBEDDING_MODEL", "qwen3-0.6b-text-embedding"),
]


def safe_collection_name(model: str) -> str:
    safe = "".join(ch if ch.isalnum() else "_" for ch in model.lower())
    return f"devtalks_embed_compare_{safe}"[:63]


def build_collection_for_model(model: str):
    start = time.perf_counter()
    test_embedding = client.embeddings.create(
        model=model,
        input="What is retrieval augmented generation?",
    ).data[0].embedding

    model_chroma = chromadb.EphemeralClient()
    model_collection = model_chroma.get_or_create_collection(
        name=safe_collection_name(model),
        embedding_function=OpenAIEmbeddingFunction(client, model),
    )
    model_collection.upsert(
        ids=[doc["id"] for doc in docs],
        documents=[doc["text"] for doc in docs],
        metadatas=[{"source": doc["source"]} for doc in docs],
    )
    elapsed = time.perf_counter() - start
    return {
        "model": model,
        "collection": model_collection,
        "dimensions": len(test_embedding),
        "index_seconds": elapsed,
    }


embedding_comparison = []
for model in EMBEDDING_MODELS_TO_COMPARE:
    try:
        result = build_collection_for_model(model)
        embedding_comparison.append(result)
        print(
            f"{model}: {result['dimensions']} dimensions, "
            f"indexed {len(docs)} docs in {result['index_seconds']:.2f}s"
        )
    except Exception as exc:
        print(f"{model}: unavailable or failed ({exc})")

Run the same questions through both indexes and compare the rankings:


In [ ]:
comparison_questions = [
    "What are the three stages of RAG?",
    "How does MCP change the RAG pipeline?",
    "What model serves embeddings in this demo?",
]


def retrieve_from_collection(model_collection, question: str, k: int = 3):
    start = time.perf_counter()
    result = model_collection.query(query_texts=[question], n_results=k)
    elapsed = time.perf_counter() - start
    hits = []
    for text, metadata, distance in zip(
        result["documents"][0],
        result["metadatas"][0],
        result["distances"][0],
    ):
        hits.append({
            "source": metadata["source"],
            "distance": distance,
            "preview": text.strip().replace("\n", " ")[:100],
        })
    return hits, elapsed


for question in comparison_questions:
    print("=" * 88)
    print("Question:", question)
    for item in embedding_comparison:
        hits, elapsed = retrieve_from_collection(item["collection"], question)
        print(f"\n{item['model']} ({item['dimensions']} dims, query {elapsed:.2f}s)")
        for rank, hit in enumerate(hits, start=1):
            print(f"  {rank}. {hit['source']} distance={hit['distance']:.4f} - {hit['preview']}...")